# Multi-Index Agentic Retrieval Demo

Demonstrates **multi-index agentic retrieval** in Azure AI Search — querying and aggregating results across multiple search indexes through a single knowledge base.

## Architecture

```
┌─────────────────────┐
│  Knowledge Base     │ ← Single KB orchestrates retrieval
│  (earth-multi-kb)   │
└──────────┬──────────┘
           │
     ┌─────┴─────┐
     │           │
┌────▼────┐ ┌───▼─────┐
│   KS-1  │ │   KS-2  │ ← Two knowledge sources
└────┬────┘ └───┬─────┘
     │          │
┌────▼────┐ ┌──▼──────┐
│ Index 1 │ │ Index 2 │ ← Two search indexes (identical schemas)
└─────────┘ └─────────┘
```

The **NASA Earth at Night** dataset is split 50/50 across two indexes to simulate data partitioned across multiple stores.

## Prerequisites

### Azure Resources
- **Azure AI Search** (Basic tier or higher) with system-assigned managed identity enabled
- **Azure OpenAI** (or Microsoft Foundry) with `text-embedding-3-large` and `gpt-4o` deployments
- A `.env` file (copy from `.env.sample`)

### Required RBAC Role Assignments

| Principal | Role | Scope | Purpose |
|---|---|---|---|
| **Your user/SP** | `Search Service Contributor` | Search service | Create/manage indexes, KBs, KSs |
| **Your user/SP** | `Search Index Data Contributor` | Search service | Upload documents to indexes |
| **Search service MI** | `Cognitive Services OpenAI User` | Azure OpenAI resource | Vectorizer + KB model access |

> **Enable RBAC auth** on the Search service: set `authOptions` to `aadOrApiKey` and `aadFailWith403` to `http401WithBearerChallenge`.
> RBAC assignments can take 5–10 minutes to propagate.

### Python Packages
```
pip install azure-search-documents>=11.7.0b2 --pre
pip install azure-identity python-dotenv requests
```

## Step 1: Environment Setup

Load environment variables and configure Azure clients. Supports both **Azure OpenAI** and **Microsoft Foundry** endpoints.

In [ ]:
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os

load_dotenv(override=True)

# Azure AI Search
search_endpoint = os.environ["SEARCH_ENDPOINT"]
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://search.azure.com/.default")

# AI Model (Azure OpenAI or Microsoft Foundry)
aoai_endpoint = os.environ["AOAI_ENDPOINT"]
aoai_embedding_model = os.environ.get("AOAI_EMBEDDING_MODEL", "text-embedding-3-large")
aoai_embedding_deployment = os.environ.get("AOAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
aoai_gpt_model = os.environ.get("AOAI_GPT_MODEL", "gpt-4")
aoai_gpt_deployment = os.environ.get("AOAI_GPT_DEPLOYMENT", "gpt-4")

# Multi-index configuration
index_name_1 = os.environ.get("INDEX_NAME_1", "index-earth-1")
index_name_2 = os.environ.get("INDEX_NAME_2", "index-earth-2")
knowledge_source_name_1 = os.environ.get("KNOWLEDGE_SOURCE_NAME_1", "earth-ks-1")
knowledge_source_name_2 = os.environ.get("KNOWLEDGE_SOURCE_NAME_2", "earth-ks-2")
knowledge_base_name = os.environ.get("KNOWLEDGE_BASE_NAME", "earth-multi-kb")

endpoint_type = "Microsoft Foundry" if "azureml" in aoai_endpoint.lower() else "Azure OpenAI"

print("\u2713 Environment configured")
print(f"  Search: {search_endpoint}")
print(f"  AI Model: {aoai_endpoint} ({endpoint_type})")
print(f"  Indexes: {index_name_1}, {index_name_2}")
print(f"  Knowledge Sources: {knowledge_source_name_1}, {knowledge_source_name_2}")
print(f"  Knowledge Base: {knowledge_base_name}")


## Step 2: Verify Authentication & Create Search Indexes

Verifies your credentials can reach Azure AI Search, then creates two indexes with identical schemas (vector search + semantic search). The vectorizer uses the Search service's managed identity to call Azure OpenAI for embeddings.

> **Forbidden errors?** Check: (1) RBAC propagation delay (up to 10 min), (2) MI not enabled on Search, (3) missing `Cognitive Services OpenAI User` role on the OpenAI resource, (4) network/firewall rules.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, VectorSearch, VectorSearchProfile,
    HnswAlgorithmConfiguration, AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters,
    SemanticSearch, SemanticConfiguration, SemanticPrioritizedFields, SemanticField
)
from azure.search.documents.indexes import SearchIndexClient
from urllib.parse import urlparse
import time

# Verify credentials
print("Verifying credentials...")
try:
    index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
    existing_indexes = {idx.name for idx in index_client.list_indexes()}
    print(f"\u2713 Authenticated. Found {len(existing_indexes)} existing indexes.")
except Exception as e:
    print(f"\u2717 Auth failed: {e}")
    raise

search_service_name = urlparse(search_endpoint).hostname.split(".")[0]
aoai_resource_name = urlparse(aoai_endpoint).hostname.split(".")[0]
print(f"  Search: {search_service_name} | AI Model: {aoai_resource_name}")

def create_search_index(index_name):
    """Create a search index with vector + semantic search."""
    return SearchIndex(
        name=index_name,
        fields=[
            SearchField(name="id", type="Edm.String", key=True, filterable=True, sortable=True, facetable=True),
            SearchField(name="page_chunk", type="Edm.String", filterable=False, sortable=False, facetable=False),
            SearchField(name="page_embedding_text_3_large", type="Collection(Edm.Single)", stored=False, vector_search_dimensions=3072, vector_search_profile_name="hnsw_text_3_large"),
            SearchField(name="page_number", type="Edm.Int32", filterable=True, sortable=True, facetable=True)
        ],
        vector_search=VectorSearch(
            profiles=[VectorSearchProfile(name="hnsw_text_3_large", algorithm_configuration_name="alg", vectorizer_name="azure_openai_text_3_large")],
            algorithms=[HnswAlgorithmConfiguration(name="alg")],
            vectorizers=[
                AzureOpenAIVectorizer(
                    vectorizer_name="azure_openai_text_3_large",
                    parameters=AzureOpenAIVectorizerParameters(
                        resource_url=aoai_endpoint,
                        deployment_name=aoai_embedding_deployment,
                        model_name=aoai_embedding_model
                    )
                )
            ]
        ),
        semantic_search=SemanticSearch(
            default_configuration_name="semantic_config",
            configurations=[
                SemanticConfiguration(
                    name="semantic_config",
                    prioritized_fields=SemanticPrioritizedFields(
                        content_fields=[
                            SemanticField(field_name="page_chunk")
                        ]
                    )
                )
            ]
        )
    )

def create_index_with_retry(index_client, index_name, max_retries=3, retry_delay=30):
    """Create or update index with retry for RBAC propagation delays."""
    if index_name in existing_indexes:
        print(f"\u2713 Index '{index_name}' already exists (skipping)")
        return
    for attempt in range(max_retries):
        try:
            index_client.create_or_update_index(create_search_index(index_name))
            print(f"\u2713 Index '{index_name}' created")
            return
        except Exception as e:
            if "Forbidden" in str(e) and attempt < max_retries - 1:
                print(f"  Attempt {attempt + 1}/{max_retries} forbidden, waiting {retry_delay}s for RBAC propagation...")
                time.sleep(retry_delay)
            else:
                print(f"\u2717 Error creating '{index_name}': {e}")
                raise

print("\nCreating search indexes...")
create_index_with_retry(index_client, index_name_1)
create_index_with_retry(index_client, index_name_2)
print("\u2713 Both indexes ready")

## Step 3: Load and Split the NASA Dataset

Fetch the NASA Earth at Night dataset from GitHub, split 50/50, and upload each half to its respective index.

In [ ]:
import requests
from azure.search.documents import SearchClient, SearchIndexingBufferedSender

print("Fetching NASA Earth at Night dataset...")
url = "https://raw.githubusercontent.com/Azure-Samples/azure-search-sample-data/refs/heads/main/nasa-e-book/earth-at-night-json/documents.json"
documents = requests.get(url).json()
total_docs = len(documents)
print(f"\u2713 Fetched {total_docs} documents")

split_point = total_docs // 2
documents_index_1 = documents[:split_point]
documents_index_2 = documents[split_point:]
print(f"  {index_name_1}: {len(documents_index_1)} docs | {index_name_2}: {len(documents_index_2)} docs")

for idx_name, docs in [(index_name_1, documents_index_1), (index_name_2, documents_index_2)]:
    # Check if index already has documents
    search_client = SearchClient(endpoint=search_endpoint, index_name=idx_name, credential=credential)
    doc_count = search_client.get_document_count()
    if doc_count >= len(docs):
        print(f"\n\u2713 '{idx_name}' already has {doc_count} documents (skipping upload)")
        continue

    print(f"\nUploading to {idx_name} ({doc_count} existing)...")
    with SearchIndexingBufferedSender(endpoint=search_endpoint, index_name=idx_name, credential=credential) as client:
        client.merge_or_upload_documents(documents=docs)
    print(f"\u2713 {len(docs)} documents upserted into '{idx_name}'")

print(f"\n\u2713 Upload complete. {total_docs} documents across 2 indexes.")

## Step 4: Create Knowledge Sources

Create two knowledge sources — each wraps one search index and defines which fields to include in retrieval results.

In [ ]:
from azure.search.documents.indexes.models import SearchIndexKnowledgeSource, SearchIndexKnowledgeSourceParameters, SearchIndexFieldReference

index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)

for ks_name, idx_name in [(knowledge_source_name_1, index_name_1), (knowledge_source_name_2, index_name_2)]:
    ks = SearchIndexKnowledgeSource(
        name=ks_name,
        description=f"Knowledge source for {idx_name}",
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=idx_name,
            source_data_fields=[
                SearchIndexFieldReference(name="id"), 
                SearchIndexFieldReference(name="page_number")
            ]
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=ks)
    print(f"\u2713 Knowledge source '{ks_name}' \u2192 {idx_name}")


## Step 5: Create the Knowledge Base

Create a single knowledge base referencing **both** knowledge sources. Configured with `ANSWER_SYNTHESIS` mode so it decomposes queries, retrieves from both indexes, and generates a cited answer.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase, KnowledgeBaseAzureOpenAIModel, KnowledgeSourceReference, 
    AzureOpenAIVectorizerParameters, KnowledgeRetrievalOutputMode
)

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=aoai_endpoint,
    deployment_name=aoai_gpt_deployment,
    model_name=aoai_gpt_model,
)

knowledge_base = KnowledgeBase(
    name=knowledge_base_name,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=knowledge_source_name_1),
        KnowledgeSourceReference(name=knowledge_source_name_2)
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Provide a concise and informative answer based on the retrieved documents. Cite which sources the information came from."
)

index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
index_client.create_or_update_knowledge_base(knowledge_base)
print(f"\u2713 Knowledge base '{knowledge_base_name}' created")
print(f"  Sources: {knowledge_source_name_1} \u2192 {index_name_1}, {knowledge_source_name_2} \u2192 {index_name_2}")
print(f"  Mode: ANSWER_SYNTHESIS | Model: {aoai_gpt_model}")


## Step 6: Execute a Multi-Index Query

Query the knowledge base with a question spanning topics from both index halves. The pipeline decomposes the query, runs subqueries against both indexes, and synthesizes a cited answer.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest, KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent, SearchIndexKnowledgeSourceParams,
    KnowledgeRetrievalLowReasoningEffort
)
import json

instructions = """You are a helpful assistant that answers questions about Earth at night.
Provide detailed answers citing the sources of information.
If information comes from multiple sources, explain how they relate."""

messages = [{"role": "system", "content": instructions}]

# Targets content spanning both index halves (urban/development in Index 1, natural phenomena in Index 2)
query = """Compare how city lights and urban development patterns (such as Las Vegas or Shanghai growth)
differ from natural light sources like volcanic eruptions, wildfires, and auroras.
What can satellite imagery of Earth at night tell us about both human expansion and natural phenomena?"""

print(f"Query: {query}\n")
messages.append({"role": "user", "content": query})

agent_client = KnowledgeBaseRetrievalClient(
    endpoint=search_endpoint,
    knowledge_base_name=knowledge_base_name,
    credential=credential
)

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role=m["role"],
            content=[KnowledgeBaseMessageTextContent(text=m["content"])]
        ) for m in messages if m["role"] != "system"
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=knowledge_source_name_1,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True  # ensures both indexes are queried
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=knowledge_source_name_2,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True
        )
    ],
    include_activity=True,
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort
)

result = agent_client.retrieve(retrieval_request=req)

response_parts = []
for resp in result.response:
    for content in resp.content:
        response_parts.append(content.text)
response_content = "\n\n".join(response_parts) if response_parts else "No response found"

activity_content = json.dumps([a.as_dict() for a in result.activity], indent=2) if result.activity else None

print(f"\u2713 Query complete")
print(f"  Response: {len(response_content)} chars | References: {len(result.references) if result.references else 0} | Activities: {len(result.activity) if result.activity else 0}")

## Step 7: Display Results with Source Attribution

Parse results and display the synthesized answer, a side-by-side comparison of references by index, and the activity log.

In [ ]:
import json

print("=" * 80)
print("SYNTHESIZED ANSWER")
print("=" * 80)
print(response_content)
print()

# Map activity IDs → knowledge source names (activity_source on references is an integer ID)
activity_id_to_ks = {}
if result.activity:
    for act in result.activity:
        act_dict = act.as_dict()
        act_id = act_dict.get('id')
        ks = (
            act_dict.get('knowledge_source_name', '') or
            act_dict.get('knowledge_source', '') or
            act_dict.get('knowledgeSourceName', '') or
            act_dict.get('source', '')
        )
        if act_id is not None and ks:
            activity_id_to_ks[act_id] = ks

ks_to_index = {
    knowledge_source_name_1: index_name_1,
    knowledge_source_name_2: index_name_2,
    index_name_1: index_name_1,
    index_name_2: index_name_2,
}

# Group references by source index
references_by_index = {index_name_1: [], index_name_2: []}

if result.references:
    for ref in result.references:
        ref_dict = ref.as_dict()
        activity_src = ref_dict.get('activity_source')
        resolved_ks = activity_id_to_ks.get(activity_src, '')
        target_index = ks_to_index.get(resolved_ks, '')

        if target_index:
            references_by_index[target_index].append(ref_dict)
        else:
            matched = False
            for v in ref_dict.values():
                if isinstance(v, str):
                    if index_name_1 in v or knowledge_source_name_1 in v:
                        references_by_index[index_name_1].append(ref_dict)
                        matched = True
                        break
                    elif index_name_2 in v or knowledge_source_name_2 in v:
                        references_by_index[index_name_2].append(ref_dict)
                        matched = True
                        break
            if not matched:
                print(f"  \u26a0 Could not determine source for reference (activity_source={activity_src!r}, resolved_ks={resolved_ks!r})")
                references_by_index[index_name_1].append(ref_dict)

print("=" * 80)
print("RESULTS BY SOURCE INDEX")
print("=" * 80)
print()

max_results = max(len(references_by_index[index_name_1]), len(references_by_index[index_name_2]))

print(f"{'INDEX 1: ' + index_name_1:^40} | {'INDEX 2: ' + index_name_2:^40}")
print("-" * 40 + " | " + "-" * 40)
print(f"Results: {len(references_by_index[index_name_1]):^33} | Results: {len(references_by_index[index_name_2]):^33}")
print("=" * 83)
print()

for i in range(max_results):
    print(f"{'RESULT ' + str(i+1):^40} | {'RESULT ' + str(i+1):^40}")
    print("-" * 40 + " | " + "-" * 40)

    if i < len(references_by_index[index_name_1]):
        ref1 = references_by_index[index_name_1][i]
        chunk1 = ref1.get('chunk', '')[:150] + "..." if len(ref1.get('chunk', '')) > 150 else ref1.get('chunk', '')
        page1 = ref1.get('source_data', {}).get('page_number', 'N/A')
        print(f"Source: {index_name_1}")
        print(f"Page: {page1}")
        print(f"Content: {chunk1}")
    else:
        print("(no more results)")

    print(" " * 40 + " | ", end="")

    if i < len(references_by_index[index_name_2]):
        ref2 = references_by_index[index_name_2][i]
        chunk2 = ref2.get('chunk', '')[:150] + "..." if len(ref2.get('chunk', '')) > 150 else ref2.get('chunk', '')
        page2 = ref2.get('source_data', {}).get('page_number', 'N/A')
        print(f"Source: {index_name_2}")
        print(f"     Page: {page2}")
        print(f"     Content: {chunk2}")
    else:
        print("(no more results)")

    print()

print("=" * 80)
print("DISTRIBUTION SUMMARY")
print("=" * 80)
total_refs = len(references_by_index[index_name_1]) + len(references_by_index[index_name_2])
if total_refs > 0:
    print(f"Total references: {total_refs}")
    print(f"  From {index_name_1}: {len(references_by_index[index_name_1])} ({len(references_by_index[index_name_1])/total_refs*100:.1f}%)")
    print(f"  From {index_name_2}: {len(references_by_index[index_name_2])} ({len(references_by_index[index_name_2])/total_refs*100:.1f}%)")
else:
    print("No references found.")

print()
print("=" * 80)
print("ACTIVITY LOG")
print("=" * 80)
if activity_content:
    print(activity_content[:800] + "..." if len(activity_content) > 800 else activity_content)
else:
    print("No activity found.")

## Step 8: Test Single-Index Access Restriction

Azure AI Search RBAC roles (`Search Index Data Reader`, `Search Index Data Contributor`) are scoped at the **service level** — they apply to all indexes in the service. To restrict a user to specific indexes, the recommended pattern is:

1. **Create a restricted knowledge base** that references only the knowledge sources the user is authorized to access
2. **Grant the user query permissions** on the search service, but point them to the restricted KB

This test creates a restricted KB referencing only `earth-ks-1` and verifies that **all results come exclusively from `index-earth-1`**.

| Scenario | Knowledge Base | Knowledge Sources | Expected Result |
|---|---|---|---|
| Full access (Step 6) | `earth-multi-kb` | `earth-ks-1` + `earth-ks-2` | Results from both indexes |
| Restricted access | `earth-restricted-kb` | `earth-ks-1` only | Results from `index-earth-1` only |


In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    KnowledgeBase, KnowledgeBaseAzureOpenAIModel, KnowledgeSourceReference,
    AzureOpenAIVectorizerParameters, KnowledgeRetrievalOutputMode
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest, KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent, SearchIndexKnowledgeSourceParams,
    KnowledgeRetrievalLowReasoningEffort
)
import json

restricted_kb_name = "earth-restricted-kb"

# Create a restricted KB with only knowledge_source_name_1
print("Creating restricted knowledge base (single knowledge source only)...")

restricted_kb = KnowledgeBase(
    name=restricted_kb_name,
    models=[KnowledgeBaseAzureOpenAIModel(
        azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
            resource_url=aoai_endpoint,
            deployment_name=aoai_gpt_deployment,
            model_name=aoai_gpt_model,
        )
    )],
    knowledge_sources=[
        KnowledgeSourceReference(name=knowledge_source_name_1)
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Provide a concise answer based on the retrieved documents. Cite sources."
)

index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
index_client.create_or_update_knowledge_base(restricted_kb)
print(f"\u2713 Restricted KB '{restricted_kb_name}' created")
print(f"  Sources: {knowledge_source_name_1} \u2192 {index_name_1} ONLY")
print(f"  Excluded: {knowledge_source_name_2} \u2192 {index_name_2}")

# Query the restricted KB with the same query
print(f"\nQuerying restricted KB with same query...")
print(f"Query: {query[:80]}...\n")

restricted_client = KnowledgeBaseRetrievalClient(
    endpoint=search_endpoint,
    knowledge_base_name=restricted_kb_name,
    credential=credential
)

restricted_req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=query)]
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=knowledge_source_name_1,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True
        )
    ],
    include_activity=True,
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort
)

restricted_result = restricted_client.retrieve(retrieval_request=restricted_req)

restricted_response = "\n".join(
    content.text for resp in restricted_result.response for content in resp.content
) or "No response"

print("=" * 80)
print("RESTRICTED KB RESPONSE (single-index access)")
print("=" * 80)
print(restricted_response)
print()

# Verify all references come from the permitted index only
restricted_refs = restricted_result.references or []
print(f"References returned: {len(restricted_refs)}")

restricted_act_map = {}
if restricted_result.activity:
    for act in restricted_result.activity:
        d = act.as_dict()
        act_id = d.get('id')
        ks = d.get('knowledge_source_name', '') or d.get('knowledge_source', '') or d.get('source', '')
        if act_id is not None and ks:
            restricted_act_map[act_id] = ks

index1_count = 0
index2_count = 0
unknown_count = 0

for ref in restricted_refs:
    ref_dict = ref.as_dict()
    activity_src = ref_dict.get('activity_source')
    resolved = restricted_act_map.get(activity_src, '')

    all_values = [str(v) for v in ref_dict.values() if v is not None]
    combined = ' '.join(all_values) + ' ' + resolved

    if knowledge_source_name_1 in combined or index_name_1 in combined:
        index1_count += 1
    elif knowledge_source_name_2 in combined or index_name_2 in combined:
        index2_count += 1
    else:
        index1_count += 1  # default — only KS in this restricted KB
        unknown_count += 1

# Verification verdict
print()
print("=" * 80)
print("RBAC RESTRICTION VERIFICATION")
print("=" * 80)
print(f"  References from {index_name_1} (permitted):   {index1_count}")
print(f"  References from {index_name_2} (restricted):  {index2_count}")
if unknown_count:
    print(f"  References with unresolved source:           {unknown_count} (attributed to permitted index)")
print()

if index2_count == 0 and index1_count > 0:
    print("\u2705 PASS: All references come from the permitted index only.")
    print("  The restricted KB correctly scoped retrieval to a single knowledge source.")
elif index2_count > 0:
    print("\u274c FAIL: References from the restricted index were returned.")
    print(f"  {index2_count} reference(s) leaked from {index_name_2}.")
else:
    print("\u26a0 WARNING: No references returned. The query may not have matched any documents in {index_name_1}.")

full_ref_count = len(result.references) if result.references else 0
print(f"\nComparison with full-access query (Step 6):")
print(f"  Full access:       {full_ref_count} references from 2 indexes")
print(f"  Restricted access: {len(restricted_refs)} references from 1 index")

if restricted_result.activity:
    print(f"\nRestricted KB Activity Log:")
    for act in restricted_result.activity:
        d = act.as_dict()
        print(f"  Activity {d.get('id','?')}: {d.get('knowledge_source_name', d.get('knowledge_source', d.get('type', '?')))}")

## Step 9: Test Index-Level RBAC with Knowledge Bases

Azure AI Search supports scoping `Search Index Data Reader` to a **single index** via PowerShell or CLI (not the portal). But does the KB retrieve API honor this?

This test creates a **service principal** with `Search Index Data Reader` scoped to only `index-earth-1`, then queries the **full multi-index KB**.

### Result

| Hypothesis | Expected Behavior | Result |
|---|---|---|
| KB uses **caller's identity** for index queries | Only `index-earth-1` results returned | Not observed |
| KB uses **service's internal identity** | Results from both indexes (RBAC bypassed) | Not observed |
| KB retrieve API requires **service-level roles** | 403 Forbidden — index-scoped role insufficient | **Confirmed** |

The KB retrieve API returned **403 Forbidden**. Index-scoped `Search Index Data Reader` is insufficient to call the KB retrieve endpoint — the caller needs a **service-level role** (e.g., `Search Index Data Reader` scoped to the entire search service, not a single index).

**Conclusion**: Index-level RBAC alone cannot scope KB retrieval. Use **KB-level scoping** (Step 8) — create separate knowledge bases per access tier — to restrict which indexes a caller can query.

> **Requires**: `Application Administrator` or equivalent Entra ID role to create a test SP. Takes ~90s for RBAC propagation. SP is deleted after the test.

In [ ]:
import time, uuid, datetime, base64, requests
from azure.identity import CertificateCredential
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa

# Acquire management and Graph tokens
mgmt_token = credential.get_token("https://management.azure.com/.default").token
graph_token = credential.get_token("https://graph.microsoft.com/.default").token
mgmt_headers = {"Authorization": f"Bearer {mgmt_token}", "Content-Type": "application/json"}
graph_headers = {"Authorization": f"Bearer {graph_token}", "Content-Type": "application/json"}

# Resolve subscription, tenant, and resource group
subs_resp = requests.get(
    "https://management.azure.com/subscriptions?api-version=2022-01-01",
    headers=mgmt_headers
)
subs_resp.raise_for_status()
sub_info = subs_resp.json()["value"][0]
sub_id = sub_info["subscriptionId"]
sp_tenant = sub_info["tenantId"]

search_resources = requests.get(
    f"https://management.azure.com/subscriptions/{sub_id}/resources"
    f"?$filter=resourceType eq 'Microsoft.Search/searchServices'"
    f" and name eq '{search_service_name}'&api-version=2021-04-01",
    headers=mgmt_headers
)
search_resources.raise_for_status()
rg_name = search_resources.json()["value"][0]["id"].split("/resourceGroups/")[1].split("/")[0]

print(f"Subscription: {sub_id}")
print(f"Resource Group: {rg_name}")
print(f"Search Service: {search_service_name}")

# Clean up orphaned app registrations from previous runs
sp_display_name = "rbac-index-test-sp"

existing_apps = requests.get(
    f"https://graph.microsoft.com/v1.0/applications?$filter=displayName eq '{sp_display_name}'",
    headers=graph_headers
).json().get("value", [])

if existing_apps:
    print(f"\nCleaning up {len(existing_apps)} orphaned app registration(s)...")
    for app in existing_apps:
        requests.delete(
            f"https://graph.microsoft.com/v1.0/applications/{app['id']}",
            headers=graph_headers
        )
        print(f"  \u2713 Deleted orphaned app: {app['appId']}")

# Generate ephemeral self-signed certificate (in memory, expires in 1 hour)
print(f"\nGenerating ephemeral certificate...")
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
subject = issuer = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, "rbac-index-test")])
cert = (
    x509.CertificateBuilder()
    .subject_name(subject)
    .issuer_name(issuer)
    .public_key(private_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.datetime.now(datetime.timezone.utc))
    .not_valid_after(datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(hours=1))
    .sign(private_key, hashes.SHA256())
)

cert_b64 = base64.b64encode(cert.public_bytes(serialization.Encoding.DER)).decode()
combined_pem = (
    private_key.private_bytes(serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8, serialization.NoEncryption())
    + cert.public_bytes(serialization.Encoding.PEM)
)
print(f"\u2713 Certificate generated (expires in 1 hour, in-memory only)")

# Create test app registration + service principal
print(f"\nCreating test SP '{sp_display_name}'...")

app_resp = requests.post(
    "https://graph.microsoft.com/v1.0/applications",
    headers=graph_headers, json={"displayName": sp_display_name}
)
if app_resp.status_code not in (200, 201):
    print(f"\u2717 App creation failed ({app_resp.status_code}): {app_resp.text}")
    raise RuntimeError("App registration failed. Ensure you have Application Administrator role.")

app_data = app_resp.json()
app_object_id = app_data["id"]
sp_app_id = app_data["appId"]
print(f"\u2713 App registered: appId={sp_app_id}")

patch_resp = requests.patch(
    f"https://graph.microsoft.com/v1.0/applications/{app_object_id}",
    headers=graph_headers,
    json={
        "keyCredentials": [{
            "type": "AsymmetricX509Cert",
            "usage": "Verify",
            "key": cert_b64,
            "displayName": "test-cert"
        }]
    }
)
if patch_resp.status_code not in (200, 204):
    print(f"\u2717 Certificate attach failed ({patch_resp.status_code}): {patch_resp.text}")
    requests.delete(f"https://graph.microsoft.com/v1.0/applications/{app_object_id}", headers=graph_headers)
    raise RuntimeError("Failed to attach certificate to app registration.")
print(f"\u2713 Certificate credential attached")

sp_resp = requests.post(
    "https://graph.microsoft.com/v1.0/servicePrincipals",
    headers=graph_headers,
    json={"appId": sp_app_id}
)
if sp_resp.status_code not in (200, 201):
    print(f"\u2717 SP creation failed ({sp_resp.status_code}): {sp_resp.text}")
    requests.delete(f"https://graph.microsoft.com/v1.0/applications/{app_object_id}", headers=graph_headers)
    raise RuntimeError("Service principal creation failed.")

sp_data = sp_resp.json()
sp_oid = sp_data["id"]
print(f"\u2713 SP created: objectId={sp_oid}")

# Assign Search Index Data Reader scoped to index-earth-1 only
index_scope = (
    f"/subscriptions/{sub_id}/resourceGroups/{rg_name}"
    f"/providers/Microsoft.Search/searchServices/{search_service_name}"
    f"/indexes/{index_name_1}"
)
print(f"\nAssigning 'Search Index Data Reader' scoped to: {index_name_1}")

role_def_id = "1407120a-92aa-4202-b7e9-c0e197c71c8f"
assignment_id = str(uuid.uuid4())
assign_resp = requests.put(
    f"https://management.azure.com{index_scope}/providers/Microsoft.Authorization"
    f"/roleAssignments/{assignment_id}?api-version=2022-04-01",
    headers=mgmt_headers,
    json={
        "properties": {
            "roleDefinitionId": f"/subscriptions/{sub_id}/providers/Microsoft.Authorization/roleDefinitions/{role_def_id}",
            "principalId": sp_oid,
            "principalType": "ServicePrincipal"
        }
    }
)
if assign_resp.status_code in (200, 201):
    print(f"\u2713 Role assigned (index-scoped only, no service-level roles)")
else:
    print(f"\u2717 Role assignment failed: {assign_resp.status_code} {assign_resp.text}")

print("\nWaiting 90s for RBAC propagation...")
time.sleep(90)

# Query the full multi-index KB as the restricted SP
print("\n" + "=" * 80)
print("TEST: Query full KB as SP with index-level RBAC on index-earth-1 only")
print("=" * 80)

sp_credential = CertificateCredential(sp_tenant, sp_app_id, certificate_data=combined_pem)

try:
    sp_client = KnowledgeBaseRetrievalClient(
        endpoint=search_endpoint,
        knowledge_base_name=knowledge_base_name,
        credential=sp_credential
    )

    sp_req = KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role="user",
                content=[KnowledgeBaseMessageTextContent(text=query)]
            )
        ],
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=knowledge_source_name_1,
                include_references=True,
                include_reference_source_data=True,
                always_query_source=True
            ),
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=knowledge_source_name_2,
                include_references=True,
                include_reference_source_data=True,
                always_query_source=True
            )
        ],
        include_activity=True,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort
    )

    sp_retrieve = sp_client.retrieve(retrieval_request=sp_req)

    sp_refs = sp_retrieve.references or []
    sp_response = "\n".join(
        c.text for r in sp_retrieve.response for c in r.content
    )
    print(f"\n\u2713 KB retrieve succeeded")
    print(f"  Response: {len(sp_response)} chars | References: {len(sp_refs)}")

    sp_act_map = {}
    if sp_retrieve.activity:
        for act in sp_retrieve.activity:
            d = act.as_dict()
            ks = d.get('knowledge_source_name', '') or d.get('knowledge_source', '') or d.get('source', '')
            if d.get('id') is not None and ks:
                sp_act_map[d['id']] = ks

    idx1_n, idx2_n = 0, 0
    for ref in sp_refs:
        rd = ref.as_dict()
        resolved = sp_act_map.get(rd.get('activity_source'), '')
        vals = ' '.join(str(v) for v in rd.values() if v) + ' ' + resolved
        if knowledge_source_name_1 in vals or index_name_1 in vals:
            idx1_n += 1
        elif knowledge_source_name_2 in vals or index_name_2 in vals:
            idx2_n += 1
        else:
            idx1_n += 1

    print(f"\n  {index_name_1} refs (RBAC granted): {idx1_n}")
    print(f"  {index_name_2} refs (no RBAC):       {idx2_n}")

    if idx2_n > 0:
        print("\n\u26a0 FINDING: KB returned results from BOTH indexes.")
        print("  The KB retrieve API queries indexes using the service's internal")
        print("  identity, NOT the caller's token. Index-level RBAC on the caller")
        print("  does NOT restrict which indexes the KB can access.")
        print("  \u2192 Use KB-level scoping (Step 8) to restrict access.")
    elif idx1_n > 0 and idx2_n == 0:
        print("\n\u2705 FINDING: KB honored index-level RBAC.")
        print("  Only the permitted index returned results.")
    else:
        print("\n\u26a0 No references returned \u2014 inconclusive.")

except Exception as e:
    error_msg = str(e)
    if "Forbidden" in error_msg or "403" in error_msg:
        print(f"\n\u2717 KB retrieve returned 403 Forbidden")
        print(f"  Index-scoped 'Search Index Data Reader' is insufficient to call")
        print(f"  the KB retrieve API. The SP needs a service-level role to access")
        print(f"  KB endpoints \u2014 index-level RBAC alone cannot scope KB retrieval.")
        print(f"  \u2192 Use KB-level scoping (Step 8) to restrict access.")
    elif "Unauthorized" in error_msg or "401" in error_msg:
        print(f"\n\u2717 Authentication failed (401)")
        print(f"  The SP may need more RBAC propagation time.")
    else:
        print(f"\n\u2717 Error: {e}")
    print(f"\n  Conclusion: Index-level RBAC alone is not viable for scoping KB")
    print(f"  retrieval. Use KB-level scoping (Step 8) instead.")

# Cleanup: delete role assignment, app registration, and SP
print("\n" + "=" * 80)
print("CLEANUP")
print("=" * 80)

mgmt_token = credential.get_token("https://management.azure.com/.default").token
graph_token = credential.get_token("https://graph.microsoft.com/.default").token

requests.delete(
    f"https://management.azure.com{index_scope}/providers/Microsoft.Authorization"
    f"/roleAssignments/{assignment_id}?api-version=2022-04-01",
    headers={"Authorization": f"Bearer {mgmt_token}"}
)
print(f"\u2713 Role assignment removed")

requests.delete(
    f"https://graph.microsoft.com/v1.0/applications/{app_object_id}",
    headers={"Authorization": f"Bearer {graph_token}"}
)
print(f"\u2713 App registration + SP '{sp_display_name}' deleted")
print("\nCertificate and private key were in-memory only and are now discarded.")

## Step 10: Cleanup

Delete all resources created by this demo. Only run when you're done — you'll need to re-run from the beginning to recreate them.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient

index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)

print("Cleaning up resources...\n")

for name, delete_fn in [
    (knowledge_base_name, index_client.delete_knowledge_base),
    ("earth-restricted-kb", index_client.delete_knowledge_base),
    (knowledge_source_name_1, index_client.delete_knowledge_source),
    (knowledge_source_name_2, index_client.delete_knowledge_source),
    (index_name_1, index_client.delete_index),
    (index_name_2, index_client.delete_index),
]:
    try:
        delete_fn(name)
        print(f"\u2713 Deleted '{name}'")
    except Exception as e:
        print(f"\u2717 Error deleting '{name}': {e}")

print("\n\u2713 Cleanup complete. Re-run all cells to recreate resources.")


## Conclusion

### Multi-Index Agentic Retrieval
1. A single knowledge base orchestrates retrieval across multiple indexes
2. The pipeline handles query decomposition, concurrent execution, and result aggregation automatically
3. References include source metadata for attribution
4. Scales to N indexes by adding more knowledge sources

### Access Control Findings

| Access Pattern | Direct Index Query | KB Retrieve API |
|---|---|---|
| **Service-level** `Search Index Data Reader` | All indexes accessible | All indexes accessible |
| **Index-scoped** `Search Index Data Reader` on `index-earth-1` only | `index-earth-1`: accessible, `index-earth-2`: blocked | **403 Forbidden** — cannot call KB API at all |
| **KB-level scoping** (restricted KB with one knowledge source) | N/A | Only the permitted index returns results |

**Key finding**: The KB retrieve API requires **service-level** RBAC roles — index-scoped roles are insufficient to even call the endpoint (403). This means:

- **For direct index queries** (without a KB): Index-level RBAC works as expected. A user scoped to `index-earth-1` can query that index but is blocked from `index-earth-2`.
- **For KB-based retrieval**: Index-level RBAC cannot selectively filter which indexes the KB queries. The caller either has access to the KB API or doesn't.

### Recommended Access Control Pattern

To restrict which indexes a user can query **through knowledge bases**:

1. **Create per-audience knowledge bases** — each KB references only the knowledge sources (indexes) that audience is authorized to access
2. **Grant users service-level `Search Index Data Reader`** — required for the KB retrieve API to function
3. **Direct users to their scoped KB** — the KB itself enforces which indexes are queried

This is the pattern demonstrated in **Step 8** (`earth-restricted-kb` with a single knowledge source), which successfully returned results only from the permitted index.

For more information, see the [Azure AI Search agentic retrieval documentation](https://learn.microsoft.com/azure/search/search-agentic-retrieval).